# Multimodal Whisper + Vision

**Talk to an image.** Upload an audio recording and a picture. Whisper turns the speech into text; BLIP generates an image caption and answers visual questions; CLIP compares the spoken text with the image.

**Recommended runtime:** Google Colab → Runtime → Change runtime type → **T4 GPU** (CPU works but is much slower).

This notebook is self-contained. It does not need a Git clone, and it rebuilds the project files in the Colab runtime as it starts.

In [ ]:
# 1. Install runtime dependencies and FFmpeg
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install --upgrade "transformers>=4.45,<5.0" "accelerate>=0.34" "safetensors>=0.4" "pillow>=10" "soundfile>=0.12" requests

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook will run on CPU, but inference will be slower.")

In [ ]:
# 2. Rebuild the project files in this Colab runtime
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/multimodal-whisper-vision")
PROJECT_FILES = {'.gitignore': '__pycache__/\n*.py[cod]\n.ipynb_checkpoints/\n.venv/\nvenv/\n*.wav\n*.mp3\n*.webm\n*.mp4\n*.egg-info/\n.DS_Store\n.env\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026 Tejeshwini89\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'README.md': '# Multimodal Whisper + Vision\n\n**Speak about an image.** This project combines Whisper speech recognition with BLIP image captioning and visual question answering, then uses CLIP to measure how closely the spoken text matches the supplied image.\n\n[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tejeshwini89/multimodal-whisper-vision/blob/main/notebooks/colab_demo.ipynb)\n\n## What it does\n\n1. **Whisper ASR** converts an audio recording into text. Use `transcribe` to retain the spoken language or `translate` to produce English text.\n2. **Intent routing** detects whether the speaker asks a visual question, requests a description, or makes a statement.\n3. **BLIP captioning** generates a general description of the image.\n4. **BLIP VQA** answers the transcribed question using the image.\n5. **CLIP** computes cosine similarity between the image and the transcript as a lightweight grounding signal.\n\n## Architecture\n\n```text\nAudio file\n   │\n   ▼\nWhisper (ASR / speech translation)\n   │\n   ▼\nTranscript ──► Intent detection ──┬── visual question ──► BLIP VQA ──┐\n                                  ├── describe image ──► BLIP caption ├──► Result\n                                  └── statement ───────► caption + text┘\n\nImage ───────────────────────────────► BLIP caption / BLIP VQA\n  │\n  └──────────────────────────────────► CLIP image-text cosine similarity\n```\n\n## Run it in Google Colab (recommended)\n\n1. Open [`notebooks/colab_demo.ipynb`](notebooks/colab_demo.ipynb) in Google Colab.\n2. Select **Runtime → Change runtime type → T4 GPU** when available. CPU is supported but slower.\n3. Run the notebook from top to bottom.\n4. Upload one audio file and one image when prompted.\n5. Read the transcript, detected intent, image caption, visual answer and CLIP similarity.\n\nThe notebook is self-contained: it does **not** require cloning the repository before running. It installs its dependencies, writes the package modules into the Colab runtime and downloads the pretrained models on first use.\n\n## Supported input\n\n- Audio: WAV, MP3, M4A, FLAC, OGG, WEBM and other formats that FFmpeg can decode.\n- Image: common formats supported by Pillow, including PNG, JPEG and WEBP.\n- Whisper tasks: `transcribe` or `translate` (speech translation to English).\n\n## Model components\n\n| Stage | Model | Purpose |\n|---|---|---|\n| Speech recognition | [`openai/whisper-small`](https://huggingface.co/openai/whisper-small) | Multilingual speech-to-text and translation |\n| Image captioning | [`Salesforce/blip-image-captioning-base`](https://huggingface.co/Salesforce/blip-image-captioning-base) | General image description |\n| Visual question answering | [`Salesforce/blip-vqa-base`](https://huggingface.co/Salesforce/blip-vqa-base) | Answers a natural-language question about an image |\n| Image-text similarity | [`openai/clip-vit-base-patch32`](https://huggingface.co/openai/clip-vit-base-patch32) | Compares the transcript and image embeddings |\n\nThe BLIP model cards currently document direct model-class usage and note that the older BLIP task pipelines are not supported in Transformers 5. The Colab notebook therefore installs `transformers` 4.x and uses the model classes directly.\n\n## Project layout\n\n```text\nmultimodal-whisper-vision/\n├── multimodal_whisper/\n│   ├── __init__.py\n│   ├── speech.py\n│   ├── vision.py\n│   └── pipeline.py\n├── notebooks/\n│   └── colab_demo.ipynb\n├── tests/\n│   └── test_intent.py\n├── app.py                 # optional local Gradio UI\n├── cli.py                 # optional command-line interface\n├── requirements.txt\n├── LICENSE\n└── README.md\n```\n\n## Optional: run locally\n\nPython 3.10+ and FFmpeg are required. A CUDA GPU is recommended.\n\n```bash\n# Install FFmpeg first (Ubuntu/Debian)\nsudo apt-get update && sudo apt-get install -y ffmpeg\n\npython -m venv .venv\nsource .venv/bin/activate       # Windows: .venv\\\\Scripts\\\\activate\npip install -r requirements.txt\npip install "gradio>=4.44"     # only if you want the optional web UI\n\npython cli.py --audio question.mp3 --image photo.jpg\npython app.py                   # optional Gradio UI\npytest -q\n```\n\n## Example output\n\n```json\n{\n  "transcript": "How many cats are in the picture?",\n  "intent": "question",\n  "image_caption": "two cats sitting on a couch",\n  "answer": "2",\n  "speech_image_similarity": {\n    "cosine_similarity": 0.27,\n    "scaled_0_to_1": 0.635\n  }\n}\n```\n\nThe values above are illustrative, not benchmark results. CLIP cosine similarity is a relative compatibility signal, **not a calibrated probability or proof that an answer is correct**. BLIP answers can also be wrong; verify important outputs against the image.\n\n## Limitations and responsible use\n\n- Whisper transcription quality varies with noise, accents, recording quality and language.\n- BLIP VQA is a general-purpose model and may miscount objects, miss small details or hallucinate.\n- CLIP similarity is not an answer-confidence score and should not be treated as one.\n- Do not use model outputs as the sole basis for medical, legal, safety-critical or other high-impact decisions.\n- Obtain consent before transcribing recordings of other people.\n- Review each pretrained model\'s card and license before redistribution or commercial deployment. The component models have their own terms, separate from this repository\'s MIT license.\n\n## Tests\n\nThe intent router can be tested without downloading neural network weights:\n\n```bash\npytest -q\n```\n\n## Acknowledgements\n\nThis project uses pretrained models published by OpenAI and Salesforce AI Research through Hugging Face Transformers. See the linked model cards for original papers, training details, limitations and model-specific terms.\n', 'requirements.txt': 'torch>=2.2\ntransformers>=4.45,<5.0\naccelerate>=0.34\nsafetensors>=0.4\npillow>=10.0\nsoundfile>=0.12\npytest>=8.0\n# Optional local Gradio interface:\n# gradio>=4.44\n', 'app.py': '"""Optional local Gradio interface. Run with: python app.py"""\nfrom __future__ import annotations\n\nimport gradio as gr\n\nfrom multimodal_whisper import MultimodalPipeline\n\n_PIPELINE = None\n\n\ndef get_pipeline():\n    global _PIPELINE\n    if _PIPELINE is None:\n        _PIPELINE = MultimodalPipeline()\n    return _PIPELINE\n\n\ndef infer(audio_path, image, task):\n    if audio_path is None or image is None:\n        return "Upload both an audio recording and an image.", "", "", ""\n    result = get_pipeline().run(audio_path, image, task=task)\n    score = result["speech_image_similarity"]["cosine_similarity"]\n    return (\n        result["transcript"],\n        result["image_caption"],\n        result["answer"],\n        f"Intent: {result[\'intent\']} | CLIP cosine similarity: {score:.4f}",\n    )\n\n\ndemo = gr.Interface(\n    fn=infer,\n    inputs=[\n        gr.Audio(sources=["upload", "microphone"], type="filepath", label="Speak or upload audio"),\n        gr.Image(type="pil", label="Image to understand"),\n        gr.Radio(["transcribe", "translate"], value="transcribe", label="Whisper task"),\n    ],\n    outputs=[\n        gr.Textbox(label="Whisper transcript"),\n        gr.Textbox(label="BLIP image caption"),\n        gr.Textbox(label="Visual answer"),\n        gr.Textbox(label="Intent and CLIP similarity"),\n    ],\n    title="Multimodal Whisper + Vision",\n    description="Speak about an image. Whisper transcribes your speech; BLIP understands the image; CLIP scores image-text similarity.",\n)\n\nif __name__ == "__main__":\n    demo.launch()\n', 'cli.py': '"""Command-line entry point. Example: python cli.py --audio question.wav --image photo.jpg"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\n\nfrom multimodal_whisper import MultimodalPipeline\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Ask questions about an image using your voice.")\n    parser.add_argument("--audio", required=True, help="Path to an audio file")\n    parser.add_argument("--image", required=True, help="Path to an image file")\n    parser.add_argument("--model", default="openai/whisper-small", help="Hugging Face Whisper model ID")\n    parser.add_argument("--task", choices=("transcribe", "translate"), default="transcribe")\n    parser.add_argument("--language", default=None, help="Optional Whisper language hint, e.g. English")\n    args = parser.parse_args()\n\n    pipeline = MultimodalPipeline(whisper_model=args.model)\n    result = pipeline.run(args.audio, args.image, language=args.language, task=args.task)\n    print(json.dumps(result, indent=2, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'publish.sh': '#!/usr/bin/env bash\n# Create the public repository and publish the current project using GitHub CLI.\nset -euo pipefail\nOWNER="Tejeshwini89"\nREPO="multimodal-whisper-vision"\nDESCRIPTION="Multimodal AI: Whisper speech-to-text with vision (BLIP, CLIP)"\n\nif ! command -v gh >/dev/null 2>&1; then\n  echo "GitHub CLI (gh) is required. Install it and run: gh auth login" >&2\n  exit 1\nfi\nif [[ "$(gh api user --jq .login)" != "$OWNER" ]]; then\n  echo "Please authenticate GitHub CLI as $OWNER first: gh auth login" >&2\n  exit 1\nfi\n\ngit init -b main\ngit add -A\ngit commit -m "Initial commit: multimodal Whisper + vision system"\ngh repo create "$OWNER/$REPO" --public --description "$DESCRIPTION" --source=. --remote=origin --push\n', 'multimodal_whisper/__init__.py': '"""Multimodal Whisper + Vision."""\nfrom .pipeline import MultimodalPipeline, detect_intent\nfrom .speech import WhisperTranscriber\nfrom .vision import VisionEngine\n\n__all__ = ["MultimodalPipeline", "WhisperTranscriber", "VisionEngine", "detect_intent"]\n__version__ = "1.0.0"\n', 'multimodal_whisper/speech.py': '"""Whisper-based transcription and speech translation."""\nfrom __future__ import annotations\n\nimport subprocess\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\n\n\nclass WhisperTranscriber:\n    """Transcribe or translate an audio file with a Hugging Face Whisper model.\n\n    Audio is normalized to mono, 16 kHz WAV with ffmpeg before inference. This\n    avoids relying on inconsistent input codecs or sample rates.\n    """\n\n    def __init__(\n        self,\n        model_name: str = "openai/whisper-small",\n        device: str | None = None,\n        chunk_length_s: int = 30,\n        stride_length_s: tuple[int, int] = (5, 5),\n    ) -> None:\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.torch_dtype = torch.float16 if self.device.startswith("cuda") else torch.float32\n        self.chunk_length_s = chunk_length_s\n        self.stride_length_s = stride_length_s\n        self.model_name = model_name\n\n        try:\n            from transformers import pipeline\n        except ImportError as exc:\n            raise RuntimeError("Install project dependencies with `pip install -r requirements.txt`.") from exc\n\n        self.asr = pipeline(\n            task="automatic-speech-recognition",\n            model=model_name,\n            device=0 if self.device.startswith("cuda") else -1,\n            torch_dtype=self.torch_dtype,\n            chunk_length_s=chunk_length_s,\n        )\n\n    @staticmethod\n    def _normalize_audio(audio_path: str | Path) -> tuple[str, tempfile.TemporaryDirectory[str]]:\n        source = Path(audio_path).expanduser()\n        if not source.is_file():\n            raise FileNotFoundError(f"Audio file not found: {source}")\n\n        temp_dir = tempfile.TemporaryDirectory(prefix="multimodal_whisper_")\n        wav_path = str(Path(temp_dir.name) / "normalized.wav")\n        command = [\n            "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",\n            "-i", str(source), "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", wav_path,\n        ]\n        try:\n            subprocess.run(command, check=True, capture_output=True, text=True)\n        except FileNotFoundError as exc:\n            temp_dir.cleanup()\n            raise RuntimeError("ffmpeg is required. Install it with `apt-get install ffmpeg`.") from exc\n        except subprocess.CalledProcessError as exc:\n            temp_dir.cleanup()\n            details = (exc.stderr or "").strip()\n            raise ValueError(f"Could not decode audio file {source.name}: {details}") from exc\n        return wav_path, temp_dir\n\n    def transcribe(\n        self,\n        audio_path: str | Path,\n        language: str | None = None,\n        task: str = "transcribe",\n    ) -> dict[str, Any]:\n        """Return transcript and chunk timestamps. `task=\'translate\'` outputs English."""\n        if task not in {"transcribe", "translate"}:\n            raise ValueError("task must be \'transcribe\' or \'translate\'")\n\n        normalized_path, temp_dir = self._normalize_audio(audio_path)\n        generate_kwargs: dict[str, Any] = {"task": task}\n        if language:\n            generate_kwargs["language"] = language\n\n        try:\n            output = self.asr(\n                normalized_path,\n                return_timestamps=True,\n                stride_length_s=self.stride_length_s,\n                generate_kwargs=generate_kwargs,\n            )\n        finally:\n            temp_dir.cleanup()\n\n        chunks = output.get("chunks") or []\n        segments = []\n        for chunk in chunks:\n            timestamp = chunk.get("timestamp")\n            segments.append({\n                "start": timestamp[0] if timestamp and len(timestamp) > 0 else None,\n                "end": timestamp[1] if timestamp and len(timestamp) > 1 else None,\n                "text": (chunk.get("text") or "").strip(),\n            })\n\n        return {\n            "text": (output.get("text") or "").strip(),\n            "segments": segments,\n            "task": task,\n            "language_hint": language,\n            "model": self.model_name,\n        }\n', 'multimodal_whisper/vision.py': '"""BLIP image understanding and CLIP image-text similarity."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\n\n\nclass VisionEngine:\n    """Load BLIP captioning, BLIP VQA and CLIP models once for repeated use."""\n\n    CAPTION_MODEL = "Salesforce/blip-image-captioning-base"\n    VQA_MODEL = "Salesforce/blip-vqa-base"\n    CLIP_MODEL = "openai/clip-vit-base-patch32"\n\n    def __init__(self, device: str | None = None) -> None:\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.torch_dtype = torch.float16 if self.device.startswith("cuda") else torch.float32\n\n        try:\n            from transformers import (\n                BlipForConditionalGeneration, BlipForQuestionAnswering,\n                BlipProcessor, CLIPModel, CLIPProcessor,\n            )\n        except ImportError as exc:\n            raise RuntimeError("Install project dependencies with `pip install -r requirements.txt`.") from exc\n\n        self.caption_processor = BlipProcessor.from_pretrained(self.CAPTION_MODEL)\n        self.caption_model = BlipForConditionalGeneration.from_pretrained(\n            self.CAPTION_MODEL, torch_dtype=self.torch_dtype\n        ).to(self.device).eval()\n\n        self.vqa_processor = BlipProcessor.from_pretrained(self.VQA_MODEL)\n        self.vqa_model = BlipForQuestionAnswering.from_pretrained(\n            self.VQA_MODEL, torch_dtype=self.torch_dtype\n        ).to(self.device).eval()\n\n        self.clip_processor = CLIPProcessor.from_pretrained(self.CLIP_MODEL)\n        self.clip_model = CLIPModel.from_pretrained(\n            self.CLIP_MODEL, torch_dtype=self.torch_dtype\n        ).to(self.device).eval()\n\n    @staticmethod\n    def load_image(image: str | Path | Image.Image) -> Image.Image:\n        if isinstance(image, Image.Image):\n            return image.convert("RGB")\n        path = Path(image).expanduser()\n        if not path.is_file():\n            raise FileNotFoundError(f"Image file not found: {path}")\n        with Image.open(path) as opened:\n            return opened.convert("RGB")\n\n    def _move_inputs(self, inputs: Any) -> Any:\n        inputs = inputs.to(self.device)\n        if "pixel_values" in inputs:\n            inputs["pixel_values"] = inputs["pixel_values"].to(dtype=self.torch_dtype)\n        return inputs\n\n    @torch.inference_mode()\n    def caption(self, image: str | Path | Image.Image, max_new_tokens: int = 40) -> str:\n        pil_image = self.load_image(image)\n        inputs = self._move_inputs(self.caption_processor(images=pil_image, return_tensors="pt"))\n        generated = self.caption_model.generate(\n            **inputs, max_new_tokens=max_new_tokens, num_beams=3\n        )\n        return self.caption_processor.decode(generated[0], skip_special_tokens=True).strip()\n\n    @torch.inference_mode()\n    def answer(self, image: str | Path | Image.Image, question: str, max_new_tokens: int = 24) -> str:\n        if not question.strip():\n            raise ValueError("The visual question cannot be empty")\n        pil_image = self.load_image(image)\n        inputs = self._move_inputs(\n            self.vqa_processor(images=pil_image, text=question.strip(), return_tensors="pt")\n        )\n        generated = self.vqa_model.generate(\n            **inputs, max_new_tokens=max_new_tokens, num_beams=2\n        )\n        return self.vqa_processor.decode(generated[0], skip_special_tokens=True).strip()\n\n    @torch.inference_mode()\n    def similarity(self, image: str | Path | Image.Image, text: str) -> dict[str, float]:\n        """Return cosine similarity; this is a ranking signal, not a probability."""\n        pil_image = self.load_image(image)\n        inputs = self._move_inputs(\n            self.clip_processor(text=[text], images=pil_image, return_tensors="pt", padding=True, truncation=True)\n        )\n        outputs = self.clip_model(**inputs)\n        image_embedding = F.normalize(outputs.image_embeds, dim=-1)\n        text_embedding = F.normalize(outputs.text_embeds, dim=-1)\n        cosine = F.cosine_similarity(image_embedding, text_embedding, dim=-1).item()\n        return {\n            "cosine_similarity": round(float(cosine), 4),\n            "scaled_0_to_1": round(float((cosine + 1.0) / 2.0), 4),\n        }\n', 'multimodal_whisper/pipeline.py': '"""Orchestrate speech transcription, intent routing and visual inference."""\nfrom __future__ import annotations\n\nimport re\nfrom typing import Any\n\nfrom .speech import WhisperTranscriber\nfrom .vision import VisionEngine\n\n\n_DESCRIBE_PATTERNS = (\n    r"\\bdescribe\\b", r"\\bcaption\\b", r"\\bwhat do you see\\b",\n    r"\\btell me about (the )?(image|picture|photo)\\b",\n    r"\\bexplain (the )?(image|picture|photo)\\b",\n)\n_QUESTION_START = re.compile(\n    r"^(?:what|who|where|when|why|how|which|whose|whom|is|are|am|was|were|do|does|did|can|could|would|will|has|have|had)\\b",\n    re.IGNORECASE,\n)\n_ACTION_START = re.compile(\n    r"^(?:count|identify|find|locate|name|list|compare|read|tell|recognize|point out)\\b",\n    re.IGNORECASE,\n)\n\n\ndef detect_intent(text: str) -> str:\n    """Classify speech as a visual question, image description, or statement."""\n    normalized = re.sub(r"\\s+", " ", (text or "")).strip()\n    if not normalized:\n        return "statement"\n    if any(re.search(pattern, normalized, flags=re.IGNORECASE) for pattern in _DESCRIBE_PATTERNS):\n        return "describe"\n    if normalized.endswith("?") or _QUESTION_START.search(normalized) or _ACTION_START.search(normalized):\n        return "question"\n    return "statement"\n\n\nclass MultimodalPipeline:\n    """Speech → Whisper → intent → BLIP/CLIP → structured result."""\n\n    def __init__(\n        self,\n        whisper_model: str = "openai/whisper-small",\n        device: str | None = None,\n    ) -> None:\n        self.speech = WhisperTranscriber(model_name=whisper_model, device=device)\n        self.vision = VisionEngine(device=device)\n\n    def run(\n        self,\n        audio_path: str,\n        image: Any,\n        language: str | None = None,\n        task: str = "transcribe",\n    ) -> dict[str, Any]:\n        if image is None:\n            raise ValueError("Please provide an image")\n        asr = self.speech.transcribe(audio_path, language=language, task=task)\n        transcript = asr["text"]\n        if not transcript:\n            raise RuntimeError("Whisper returned an empty transcript. Try a clearer or longer recording.")\n\n        intent = detect_intent(transcript)\n        caption = self.vision.caption(image)\n        if intent == "question":\n            answer = self.vision.answer(image, transcript)\n        elif intent == "describe":\n            answer = caption\n        else:\n            answer = f"I heard: {transcript}\\nImage description: {caption}"\n\n        similarity = self.vision.similarity(image, transcript)\n        return {\n            "transcript": transcript,\n            "segments": asr["segments"],\n            "task": asr["task"],\n            "intent": intent,\n            "image_caption": caption,\n            "answer": answer,\n            "speech_image_similarity": similarity,\n            "models": {\n                "speech": asr["model"],\n                "captioning": self.vision.CAPTION_MODEL,\n                "visual_question_answering": self.vision.VQA_MODEL,\n                "image_text_similarity": self.vision.CLIP_MODEL,\n            },\n        }\n', 'tests/test_intent.py': 'from multimodal_whisper.pipeline import detect_intent\n\n\ndef test_question_with_question_mark():\n    assert detect_intent("What color is the car?") == "question"\n\n\ndef test_question_without_question_mark():\n    assert detect_intent("How many people are in the picture") == "question"\n\n\ndef test_describe_request():\n    assert detect_intent("Please describe the image") == "describe"\n\n\ndef test_describe_what_do_you_see():\n    assert detect_intent("What do you see in this photo?") == "describe"\n\n\ndef test_action_request():\n    assert detect_intent("Count the number of dogs") == "question"\n\n\ndef test_statement():\n    assert detect_intent("Nice weather today") == "statement"\n\n\ndef test_empty_text():\n    assert detect_intent("") == "statement"\n\n\ndef test_substring_does_not_trigger_describe():\n    assert detect_intent("The cat is sitting") == "statement"\n'}

for relative_path, content in PROJECT_FILES.items():
    target = PROJECT_ROOT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project workspace ready: {PROJECT_ROOT}")
print(f"Created {len(PROJECT_FILES)} source and project files.")

## 3. Upload an audio recording and an image

Upload exactly one audio file and one image in the same dialog. Whisper accepts common audio formats supported by FFmpeg; images are loaded with Pillow.

In [ ]:
from google.colab import files
from IPython.display import display, Audio
from PIL import Image

uploaded = files.upload()
AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".webm", ".aac", ".wma", ".mp4"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

audio_files = [name for name in uploaded if Path(name).suffix.lower() in AUDIO_EXTENSIONS]
image_files = [name for name in uploaded if Path(name).suffix.lower() in IMAGE_EXTENSIONS]
if len(audio_files) != 1 or len(image_files) != 1:
    raise ValueError(
        f"Please upload exactly one audio file and one image. Found {len(audio_files)} audio and {len(image_files)} image files."
    )

AUDIO_PATH = str(Path("/content") / audio_files[0])
IMAGE_PATH = str(Path("/content") / image_files[0])
image = Image.open(IMAGE_PATH).convert("RGB")
print("Audio:", AUDIO_PATH)
print("Image:", IMAGE_PATH)
display(image.resize((min(800, image.width), int(image.height * min(800, image.width) / image.width))))

## 4. Configure Whisper

`transcribe` keeps speech in its original language. `translate` asks Whisper to translate speech into English. For non-English questions, use `translate` so the BLIP visual question-answering model receives an English question. The optional language hint can help on difficult clips; leave it as `None` for automatic language detection.

In [ ]:
# Change these values if required
WHISPER_MODEL = "openai/whisper-small"
WHISPER_TASK = "transcribe"  # "transcribe" or "translate"
WHISPER_LANGUAGE = None      # e.g. "English", "Hindi", "Spanish"; None = auto-detect

from multimodal_whisper import MultimodalPipeline

pipeline = MultimodalPipeline(whisper_model=WHISPER_MODEL)
print("Whisper, BLIP captioning, BLIP VQA and CLIP are loaded.")

## 5. Run the multimodal pipeline

The spoken request is routed automatically:
- **question** → BLIP visual question answering
- **describe** → BLIP image caption
- **statement** → transcript plus image caption

The CLIP cosine similarity is a compatibility signal, not a confidence probability.

In [ ]:
import json
from IPython.display import Markdown, display

result = pipeline.run(
    audio_path=AUDIO_PATH,
    image=IMAGE_PATH,
    language=WHISPER_LANGUAGE,
    task=WHISPER_TASK,
)

print("TRANSCRIPT")
print(result["transcript"])
print("\nDETECTED INTENT")
print(result["intent"])
print("\nIMAGE CAPTION")
print(result["image_caption"])
print("\nVISUAL ANSWER")
print(result["answer"])
print("\nCLIP IMAGE–TEXT SIMILARITY")
print("Cosine similarity:", result["speech_image_similarity"]["cosine_similarity"])
print("Scaled 0–1 (not a probability):", result["speech_image_similarity"]["scaled_0_to_1"])

print("\nTIMESTAMPED SEGMENTS")
for segment in result["segments"]:
    print(segment)

print("\nMODEL CONFIGURATION")
print(json.dumps(result["models"], indent=2))

## 6. Save the result as JSON

The output file contains the transcript, timestamps, detected intent, caption, answer, model identifiers and CLIP similarity.

In [ ]:
import json
from google.colab import files

REPORT_PATH = "/content/multimodal_result.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
files.download(REPORT_PATH)